# 08 — Label Propagation

No fine-tuning loop: embed the labeled seed (5%) and the unlabeled pool with
MiniLM, build a k-NN graph over all of them, and propagate the seed labels
through the graph via `sklearn.semi_supervised.LabelSpreading`. See
`docs/semi_supervised_methods.md` for why this method was chosen over the
other candidates (SetFit, FlexMatch, Noisy Student, co-training, ...).

Reuses the `labeled.parquet` / `unlabeled.parquet` split from
`00_data_transform.ipynb` (same split `04_weak_supervision.ipynb` and
`05_pseudo_labeling.ipynb` use). It computes its own MiniLM embeddings
rather than reusing `01_embeddings.ipynb`'s cache, since that cache is keyed
to a different stratified sample of the full training set, not this
labeled/unlabeled split.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.append(str(REPO_ROOT))

import json

import pandas as pd

from utils import config
from utils.data import stratified_sample
from utils.embeddings import get_sentence_embeddings
from utils.label_propagation import run_label_propagation
from utils.metrics import evaluate_label_quality

In [2]:
labeled_df = pd.read_parquet(config.PROCESSED_DIR / "labeled.parquet")
unlabeled_df = pd.read_parquet(config.PROCESSED_DIR / "unlabeled.parquet")
test_clean = pd.read_parquet(config.PROCESSED_DIR / "test_clean.parquet")

# labeled_df is already the small 5% seed (~6000 rows) — no further
# subsampling needed. Cap the unlabeled pool at SAMPLE_SIZE like the other
# dev-scale notebooks (this is the cheap graph-algebra method, so this cap
# is generous relative to the fine-tuning notebooks' CLASSIFIER_SAMPLE_SIZE).
unlabeled_sample = stratified_sample(unlabeled_df, config.SAMPLE_SIZE, seed=config.SEED, label_col="true_label")

overlap = set(unlabeled_sample["text"]) & set(test_clean["text"])
assert len(overlap) == 0, f"{len(overlap)} rows leaked between train pool and test set"
print(f"Labeled seed: {len(labeled_df)} | Unlabeled sample: {len(unlabeled_sample)} | Test: {len(test_clean)}")

Labeled seed: 6000 | Unlabeled sample: 8000 | Test: 7600


In [3]:
cache_name = f"minilm_labelprop_L{len(labeled_df)}_U{len(unlabeled_sample)}"
combined_texts = labeled_df["text"].tolist() + unlabeled_sample["text"].tolist()
combined_embeddings = get_sentence_embeddings(combined_texts, cache_name)

labeled_embeddings = combined_embeddings[:len(labeled_df)]
unlabeled_embeddings = combined_embeddings[len(labeled_df):]
assert unlabeled_embeddings.shape[0] == len(unlabeled_sample)
print(f"Embedded {len(combined_texts)} texts (MiniLM, {combined_embeddings.shape[1]}-d)")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/219 [00:00<?, ?it/s]

Embedded 14000 texts (MiniLM, 384-d)


In [4]:
predicted_labels, confidence = run_label_propagation(
    labeled_embeddings, labeled_df["label"].to_numpy(), unlabeled_embeddings,
    kernel="knn", n_neighbors=7)

label_quality = evaluate_label_quality(
    true_labels=unlabeled_sample["true_label"].to_numpy(),
    pseudo_labels=predicted_labels,
    confidence_scores=confidence)
print("Label propagation quality:", label_quality)

config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
with open(config.RESULTS_DIR / "metrics_label_propagation.json", "w") as f:
    json.dump(label_quality, f, indent=2)
print("Saved label propagation results.")

Label propagation quality: {'Label Accuracy': 0.871875, 'Label Macro F1': 0.8715042293813033, 'Coverage': np.float64(1.0), 'Mean Confidence': 0.9060713303062158, 'Median Confidence': 0.9920743174798577}
Saved label propagation results.
